# جلسه ۴: امبدینگ‌های متنی و جستجوی معنایی

## اهداف
- درک اینکه امبدینگ‌های متنی چیستند و چرا اهمیت دارند
- تولید امبدینگ با استفاده از API اوپن‌ای‌آی
- محاسبه شباهت بین متن‌ها
- ساخت موتور جستجوی معنایی از صفر

**مدت زمان:** ۴۰ دقیقه | **سطح:** متوسط

**چرا این مهم است:** امبدینگ‌ها پایه RAG، سیستم‌های توصیه و اپلیکیشن‌های مبتنی بر شباهت هستند.

In [ ]:
!pip install openai numpy python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# بارگذاری متغیرهای محیطی از فایل .env
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"راه‌اندازی کامل شد! مدل: {MODEL}")

Setup complete! Model: gpt-5-mini


## ۱. امبدینگ چیست؟

**امبدینگ** یک بردار (لیستی از اعداد) است که *معنای* متن را نمایش می‌دهد.

- متن‌های مشابه → بردارهایی که به هم نزدیک‌اند
- متن‌های متفاوت → بردارهایی که از هم دور هستند
- ابعاد ویژگی‌های معنایی را ثبت می‌کنند (موضوع، احساسات، سبک و غیره)

```
"I love dogs"     → [0.12, -0.34, 0.56, ...] (۱۵۳۶ بُعد)
"I adore puppies"  → [0.11, -0.33, 0.57, ...] (بسیار مشابه!)
"Quantum physics"  → [-0.45, 0.78, -0.12, ...] (بسیار متفاوت!)
```

In [ ]:
# تولید امبدینگ برای یک متن
response = client.embeddings.create(
    model="text-embedding-3-small",  # سریع و مقرون‌به‌صرفه
    input="Machine learning is a subset of artificial intelligence."
)

embedding = response.data[0].embedding

print(f"ابعاد امبدینگ: {len(embedding)}")
print(f"۱۰ مقدار اول: {embedding[:10]}")
print(f"نوع: {type(embedding)}")

NotFoundError: Error code: 404 - {'error': {'message': 'Not Found', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
# تابع کمکی برای دریافت امبدینگ‌ها
def get_embedding(text, model="text-embedding-3-small"):
    """دریافت بردار امبدینگ برای یک رشته متن."""
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding

# دریافت امبدینگ برای چندین متن به صورت یکجا (کارآمدتر)
def get_embeddings(texts, model="text-embedding-3-small"):
    """دریافت امبدینگ‌ها برای لیستی از متن‌ها در یک فراخوانی API."""
    response = client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]

print("توابع کمکی تعریف شدند!")

Helper functions defined!


## ۲. اندازه‌گیری شباهت با شباهت کسینوسی

**شباهت کسینوسی** زاویه بین دو بردار را اندازه‌گیری می‌کند:
- `1.0` = معنای یکسان
- `0.0` = نامرتبط
- `-1.0` = معنای مخالف

فرمول: $\cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}$

In [ ]:
def cosine_similarity(vec_a, vec_b):
    """محاسبه شباهت کسینوسی بین دو بردار."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# مقایسه متن‌های مشابه و متفاوت
texts = [
    "I love programming in Python",       # متن ۰
    "Python coding is my favorite hobby",  # متن ۱ (مشابه ۰)
    "The weather is sunny today",          # متن ۲ (موضوع متفاوت)
]

embeddings = get_embeddings(texts)

# مقایسه همه جفت‌ها
print("امتیازات شباهت:")
print(f"  'programming in Python' در مقابل 'Python coding': {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"  'programming in Python' در مقابل 'sunny weather': {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print(f"  'Python coding' در مقابل 'sunny weather':         {cosine_similarity(embeddings[1], embeddings[2]):.4f}")

NotFoundError: Error code: 404 - {'error': {'message': 'Not Found', 'type': 'invalid_request_error', 'param': None, 'code': None}}

## ۳. ساخت موتور جستجوی معنایی

ایده ساده است:
1. **ایندکس**: امبد کردن همه اسناد
2. **جستجو**: امبد کردن پرسش، یافتن نزدیک‌ترین اسناد
3. **بازگشت**: بازگرداندن مشابه‌ترین اسناد

In [ ]:
# پایگاه دانش ما — مجموعه‌ای از حقایق
knowledge_base = [
    "Python was created by Guido van Rossum and released in 1991.",
    "Machine learning models learn patterns from data without being explicitly programmed.",
    "The Transformer architecture was introduced in the 'Attention Is All You Need' paper in 2017.",
    "GPT stands for Generative Pre-trained Transformer.",
    "Neural networks are inspired by the structure of biological neurons in the brain.",
    "Docker containers package applications with their dependencies for consistent deployment.",
    "REST APIs use HTTP methods like GET, POST, PUT, DELETE for communication.",
    "PostgreSQL is an open-source relational database management system.",
    "Large Language Models are trained on massive amounts of text data from the internet.",
    "Fine-tuning adapts a pre-trained model to a specific task using domain-specific data."
]

# مرحله ۱: امبد کردن همه اسناد (این مرحله «ایندکس» است)
kb_embeddings = get_embeddings(knowledge_base)
print(f"{len(knowledge_base)} سند ایندکس شد")
print(f"هر امبدینگ {len(kb_embeddings[0])} بُعد دارد")

NotFoundError: Error code: 404 - {'error': {'message': 'Not Found', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
def semantic_search(query, documents, doc_embeddings, top_k=3):
    """جستجوی مرتبط‌ترین اسناد برای یک پرسش."""
    # مرحله ۲: امبد کردن پرسش
    query_embedding = get_embedding(query)
    
    # مرحله ۳: محاسبه شباهت با همه اسناد
    similarities = [
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]
    
    # مرحله ۴: مرتب‌سازی بر اساس شباهت و بازگرداندن نتایج برتر
    scored_docs = list(zip(similarities, documents))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    return scored_docs[:top_k]

# تست موتور جستجو
query = "How do transformers work?"
results = semantic_search(query, knowledge_base, kb_embeddings)

print(f"پرسش: '{query}'\n")
for score, doc in results:
    print(f"  [{score:.4f}] {doc}")

NameError: name 'kb_embeddings' is not defined

In [ ]:
# پرسش‌های مختلف را امتحان کنید — توجه کنید جستجوی معنایی چگونه معنا را درک می‌کند!
queries = [
    "What programming language was made by Guido?",
    "How to deploy applications reliably?",
    "How are AI models trained?"
]

for query in queries:
    results = semantic_search(query, knowledge_base, kb_embeddings, top_k=2)
    print(f"\nپرسش: '{query}'")
    for score, doc in results:
        print(f"  [{score:.4f}] {doc}")

NameError: name 'kb_embeddings' is not defined

## ۴. جستجوی معنایی در مقابل جستجوی کلمه کلیدی

جستجوی معنایی نتایج را بر اساس **معنا** پیدا می‌کند، نه فقط تطبیق کلمات کلیدی.

In [ ]:
# جستجوی کلمه کلیدی (ساده اما محدود)
def keyword_search(query, documents, top_k=3):
    """جستجوی ساده تطبیق کلمه کلیدی."""
    query_words = set(query.lower().split())
    scored = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

# این پرسش کلمات متفاوتی از هر سند استفاده می‌کند
query = "How do neural networks learn?"

print("=== جستجوی کلمه کلیدی ===")
for score, doc in keyword_search(query, knowledge_base):
    print(f"  [تطبیق: {score}] {doc}")

print("\n=== جستجوی معنایی ===")
for score, doc in semantic_search(query, knowledge_base, kb_embeddings):
    print(f"  [{score:.4f}] {doc}")

=== Keyword Search ===
  [matches: 2] Neural networks are inspired by the structure of biological neurons in the brain.
  [matches: 0] Python was created by Guido van Rossum and released in 1991.
  [matches: 0] Machine learning models learn patterns from data without being explicitly programmed.

=== Semantic Search ===


NameError: name 'kb_embeddings' is not defined

## تمرین: سیستم جستجوی پرسش‌های متداول (FAQ)

یک سیستم جستجوی FAQ بسازید که مرتبط‌ترین پاسخ را به سؤال کاربر پیدا کند.

In [ ]:
# پایگاه داده FAQ
faqs = [
    {"q": "How do I reset my password?", "a": "Go to Settings > Security > Reset Password and follow the instructions."},
    {"q": "What payment methods do you accept?", "a": "We accept Visa, MasterCard, PayPal, and bank transfers."},
    {"q": "How can I cancel my subscription?", "a": "Navigate to Account > Subscription > Cancel. Your access continues until the end of the billing period."},
    {"q": "Do you offer a free trial?", "a": "Yes! We offer a 14-day free trial with full access to all features."},
    {"q": "How do I contact support?", "a": "You can reach us via email at support@example.com or use the live chat on our website."},
    {"q": "What is your refund policy?", "a": "We offer a 30-day money-back guarantee. Contact support for refunds."},
]

# امبد کردن همه سؤالات FAQ
faq_questions = [faq["q"] for faq in faqs]
faq_embeddings = get_embeddings(faq_questions)

def search_faq(user_question):
    """یافتن مرتبط‌ترین پاسخ FAQ."""
    results = semantic_search(user_question, faq_questions, faq_embeddings, top_k=1)
    best_score, best_question = results[0]
    # یافتن FAQ مطابق
    for faq in faqs:
        if faq["q"] == best_question:
            return faq["a"], best_score

# تست با سؤالات کاربر (با عبارت‌بندی متفاوت از FAQها!)
test_questions = [
    "I forgot my login credentials",
    "Can I pay with PayPal?",
    "I want to stop my subscription",
    "Can I try before buying?"
]

for q in test_questions:
    answer, score = search_faq(q)
    print(f"سؤال: {q}")
    print(f"پاسخ: {answer} (اطمینان: {score:.4f})")
    print()

NotFoundError: Error code: 404 - {'error': {'message': 'Not Found', 'type': 'invalid_request_error', 'param': None, 'code': None}}

## خلاصه

**آنچه یاد گرفتید:**
- امبدینگ‌ها متن را به بردارهای عددی تبدیل می‌کنند که معنا را ثبت می‌کنند
- شباهت کسینوسی اندازه‌گیری می‌کند که دو متن چقدر مشابه هستند
- جستجوی معنایی نتایج مرتبط را حتی با عبارت‌بندی متفاوت پیدا می‌کند
- الگوی ایندکس-سپس-جستجو پایه بسیاری از اپلیکیشن‌های AI است

**جلسه بعدی:** امبدینگ‌ها را با تولید LLM ترکیب می‌کنیم تا RAG (تولید تقویت‌شده با بازیابی) بسازیم!